In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# --- Setup ---
sfreq = 1000   # Sampling rate in Hz
duration = 1.0  # seconds
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# --- Step 1: Generate low-frequency amplitude envelope (LF amp) ---
lf_freq = 4
lf_signal = np.sin(2 * np.pi * lf_freq * times)
lf_amp = np.abs(hilbert(lf_signal))  # LF amplitude envelope

# --- Step 2: Generate amp_mod(t) from word onsets ---
word_onsets = np.zeros(n_samples)
word_locs = np.random.randint(100, 900, size=5)  # simulate word onset events
word_onsets[word_locs] = 1

kernel_width = int(0.1 * sfreq)  # 100 ms width
kernel = np.exp(-np.linspace(-1, 1, 2 * kernel_width)**2 / 0.05)
kernel = kernel / kernel.max()
amp_mod = np.convolve(word_onsets, kernel, mode='same')
amp_mod = 0.25 + 0.75 * (amp_mod - amp_mod.min()) / (amp_mod.max() - amp_mod.min())  # normalize to [0.25, 1]

# --- Step 3: Generate independent HF amplitude envelope hf_amp_ind(t) ---
hf_amp_ind = np.zeros(n_samples)
for _ in range(5):
    f_i = np.random.uniform(0.05, 0.5)  # very slow fluctuations
    phi_i = np.random.uniform(-2*np.pi, 2*np.pi)
    hf_amp_ind += np.cos(2 * np.pi * f_i * times + phi_i)
hf_amp_ind = (hf_amp_ind - hf_amp_ind.min()) / (hf_amp_ind.max() - hf_amp_ind.min())


# TODO: use lf_amp to modulate hf_amp (kernel) #


# --- Step 4: Compute AAC predictor from LF_amp × HF_amp_ind ---
aac_predictor = amp_mod * (lf_amp * hf_amp_ind)
aac_predictor = (aac_predictor - np.min(aac_predictor)) / (np.max(aac_predictor) - np.min(aac_predictor))  # normalize


# --- Step 5: Final HF envelope (AAC coupling with amp_mod blending) ---
hf_envelope = (1 - amp_mod) * hf_amp_ind + amp_mod * aac_predictor



# --- Step 6: Generate HF carrier and EEG signal ---
hf_freq = 80
hf_carrier = np.sin(2 * np.pi * hf_freq * times)
hf_signal = hf_envelope * hf_carrier + 0.3 * np.random.randn(n_samples)

# --- Plot ---
plt.figure(figsize=(10, 4))
plt.plot(times, hf_signal, color='lightgrey', label='Simulated AAC EEG')
plt.plot(times, hf_envelope, label='AAC Envelope', alpha=0.7)
plt.plot(times, amp_mod, label='amp_mod(t)', linestyle='-')
plt.plot(times, aac_predictor, label='LF_amp × HF_amp_ind', linestyle=':')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('Data-Inspired AAC Simulation (Weissbart-style)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge

# --- Function: Lagged Design Matrix ---
def create_lagged_matrix(predictor, lags):
    """Return a [n_timepoints, n_lags] lagged design matrix."""
    n_times = len(predictor)
    X_lagged = np.zeros((n_times, len(lags)))
    for i, lag in enumerate(lags):
        if lag < 0:
            X_lagged[:, i] = np.concatenate([np.zeros(-lag), predictor[:n_times+lag]])
        elif lag > 0:
            X_lagged[:, i] = np.concatenate([predictor[lag:], np.zeros(lag)])
        else:
            X_lagged[:, i] = predictor
    return X_lagged

# --- Parameters ---
sfreq = 1000  # Hz
lag_times = np.arange(-100, 401, 10)  # in ms
lags = (lag_times / 1000 * sfreq).astype(int)  # convert ms to samples

# --- Create Design Matrix (AAC predictor) ---
X_aac = create_lagged_matrix(aac_predictor, lags)  # shape: [n_times, n_lags]
y = hf_signal  # EEG signal

# --- Trim edges to avoid padding artifacts ---
trim = np.max(np.abs(lags))
X_trimmed = X_aac[trim:-trim]
y_trimmed = y[trim:-trim]


# --- Fit Ridge Regression ---
alpha = 1.0  # regularization parameter
model = Ridge(alpha=alpha, fit_intercept=False)
model.fit(X_trimmed, y_trimmed)

# --- Extract TRF weights ---
trf_weights = model.coef_  # shape: [n_lags]

# --- Plot TRF ---
plt.figure(figsize=(6, 4))
plt.plot(lag_times, trf_weights, label='TRF (AAC predictor)')
plt.axvline(0, color='k', linestyle='--')
plt.xlabel('Lag (ms)')
plt.ylabel('TRF weight')
plt.title('Estimated mTRF for AAC Predictor')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
X_lf = create_lagged_matrix(lf_amp, lags)
X_hf = create_lagged_matrix(hf_amp_ind, lags)
X_aac = create_lagged_matrix(aac_predictor, lags)
X_all = np.concatenate([X_lf, X_hf, X_aac], axis=1)

model = Ridge(alpha=1.0)
model.fit(X_all, y)

trf_lf, trf_hf, trf_aac = np.split(model.coef_, 3)

# Plot them
plt.figure(figsize=(7, 4))
plt.plot(lag_times, trf_lf, label='LF_amp')
plt.plot(lag_times, trf_hf, label='HF_amp_ind')
plt.plot(lag_times, trf_aac, label='AAC (LF × HF)')
plt.axvline(0, color='k', linestyle='--')
plt.xlabel('Lag (ms)')
plt.ylabel('TRF weight')
plt.title('mTRF Weights for AAC Components')
plt.legend()
plt.tight_layout()
plt.show()


# Permutation Test

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from scipy.stats import zscore

# --- Setup ---
n_permutations = 1000
alpha = 1.0
sfreq = 1000
lag_times = np.arange(-100, 401, 10)
lags = (lag_times / 1000 * sfreq).astype(int)

# --- Design matrix: real AAC predictor ---
X = create_lagged_matrix(aac_predictor, lags)
y = hf_signal
trim = np.max(np.abs(lags))
X_trimmed = X[trim:-trim]
y_trimmed = y[trim:-trim]

# --- Fit original model ---
model = Ridge(alpha=alpha, fit_intercept=False)
model.fit(X_trimmed, y_trimmed)
trf_real = model.coef_

# --- Fit null models with permuted predictors ---
trf_nulls = np.zeros((n_permutations, len(lags)))

for i in range(n_permutations):
    aac_shuffled = np.random.permutation(aac_predictor)
    X_null = create_lagged_matrix(aac_shuffled, lags)
    X_null_trim = X_null[trim:-trim]

    model.fit(X_null_trim, y_trimmed)
    trf_nulls[i, :] = model.coef_

# --- Compute z-score and p-values ---
trf_null_mean = np.mean(trf_nulls, axis=0)
trf_null_std = np.std(trf_nulls, axis=0)
z_vals = (trf_real - trf_null_mean) / trf_null_std
p_vals = 2 * (1 - np.abs(np.array([np.mean(trf_nulls[:, i] > trf_real[i]) for i in range(len(lags))])))

# --- Optional: FDR thresholding (Benjamini–Hochberg) ---
from statsmodels.stats.multitest import multipletests
reject, p_vals_fdr, _, _ = multipletests(p_vals, alpha=0.05, method='fdr_bh')

# --- Plot ---
plt.figure(figsize=(8, 4))
plt.plot(lag_times, trf_real, label='TRF (AAC)', color='blue')
plt.fill_between(lag_times,
                 np.percentile(trf_nulls, 2.5, axis=0),
                 np.percentile(trf_nulls, 97.5, axis=0),
                 color='gray', alpha=0.3, label='Permutation 95% CI')
plt.scatter(lag_times[reject], trf_real[reject], color='red', label='FDR p < 0.05')
plt.axvline(0, color='k', linestyle='--')
plt.xlabel('Lag (ms)')
plt.ylabel('TRF weight')
plt.title('mTRF for AAC Predictor with Permutation Test')
plt.legend()
plt.tight_layout()
plt.show()
